<a href="https://colab.research.google.com/github/jamesemcnally/critical-listener/blob/main/recommender_3_average_embeddings_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Mounted at /content/drive
GPU available: True
Device: NVIDIA L4


In [4]:
import pandas as pd
import numpy as np

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# Load review-level dataset and embeddings
df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")
embeddings_masked = np.load(f"{DRIVE_DIR}/nomic_masked_with_prefix.npy")
assert embeddings_masked.shape[0] == len(df), "Embeddings/df row mismatch"

print(f"Loaded: {len(df):,} reviews | Embeddings: {embeddings_masked.shape}")

Loaded: 48,189 reviews | Embeddings: (48189, 768)


## Build album embeddings

In [5]:
def build_album_embeddings(df, embeddings, key_cols=("artist_norm", "album_norm"), length_flag_threshold=0.30):
    groups = df.groupby(list(key_cols)).indices
    album_keys, n_reviews, sources, length_flags = [], [], [], []
    album_embeddings = np.zeros((len(groups), embeddings.shape[1]), dtype=embeddings.dtype)

    for i, (key, row_positions) in enumerate(groups.items()):
        # Average review embeddings for this album, then re-normalize to unit length
        vecs = embeddings[row_positions]
        mean_vec = vecs.mean(axis=0)
        album_embeddings[i] = mean_vec / np.linalg.norm(mean_vec)
        album_keys.append(key)

        rows = df.iloc[row_positions]
        n_reviews.append(len(rows))
        sources.append(sorted(rows["dataset"].unique().tolist()))

        # Flag albums where a non-PF review is disproportionately short vs. the PF review
        pf_wc = rows.loc[rows["dataset"] == "pitchfork", "word_count"]
        other_wc = rows.loc[rows["dataset"] != "pitchfork", "word_count"]
        if len(rows) > 1 and not pf_wc.empty and not other_wc.empty:
            ratio = other_wc.min() / pf_wc.iloc[0]
            length_flags.append(ratio < length_flag_threshold)
        else:
            length_flags.append(False)

    album_df = pd.DataFrame(album_keys, columns=list(key_cols))
    album_df["n_reviews"] = n_reviews
    album_df["sources"] = sources
    album_df["length_flag"] = length_flags
    return album_df, album_embeddings


album_df, album_embeddings = build_album_embeddings(df, embeddings_masked)
print(f"Albums: {album_df.shape[0]:,} | Flagged: {album_df['length_flag'].sum()}")

Albums: 44,716 | Flagged: 235


In [ ]:
# Save album-level database to Drive
np.save(f"{DRIVE_DIR}/album_embeddings_masked.npy", album_embeddings)
album_df.to_parquet(f"{DRIVE_DIR}/album_keys_masked.parquet")

## Recommender

In [6]:
def recommend_from_album_embeddings(query_artist_norm, query_album_norm, album_df, album_embeddings, k=5):
    query_mask = (album_df["artist_norm"] == query_artist_norm) & (album_df["album_norm"] == query_album_norm)
    if not query_mask.any():
        print(f"Not found: {query_artist_norm} — {query_album_norm}")
        return None

    query_idx = album_df[query_mask].index[0]
    query_vec = album_embeddings[query_idx]

    # Dot product against all album embeddings = cosine similarity (vectors are unit-normalized)
    sims = album_embeddings @ query_vec
    sims[query_idx] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen_artists = {query_artist_norm}
    results = []
    for idx in ranked:
        artist = album_df.loc[idx, "artist_norm"]
        if artist in seen_artists:
            continue
        results.append({
            "artist": album_df.loc[idx, "artist_norm"],
            "album": album_df.loc[idx, "album_norm"],
            "score": round(float(sims[idx]), 4),
            "n_reviews": album_df.loc[idx, "n_reviews"],
            "length_flag": album_df.loc[idx, "length_flag"]
        })
        seen_artists.add(artist)
        if len(results) >= k:
            break
    return results

In [ ]:
# Test query
recs = recommend_from_album_embeddings("radiohead", "kid a", album_df, album_embeddings)
for r in recs:
    print(r)

{'artist': 'endochine', 'album': 'day two', 'score': 0.8478, 'n_reviews': np.int64(1), 'length_flag': np.False_}
{'artist': 'idiot pilot', 'album': 'strange we should meet here', 'score': 0.846, 'n_reviews': np.int64(1), 'length_flag': np.False_}
{'artist': 'muse', 'album': 'showbiz', 'score': 0.8401, 'n_reviews': np.int64(1), 'length_flag': np.False_}
{'artist': 'aereogramme', 'album': 'sleep and release', 'score': 0.8386, 'n_reviews': np.int64(1), 'length_flag': np.False_}
{'artist': 'thom yorke', 'album': 'the eraser', 'score': 0.8381, 'n_reviews': np.int64(2), 'length_flag': np.True_}


## Generate recommendation datasets:

1. From all the albums in our dataset
2. From a curated cumulative list of the top 10 Billboard albums per year, 2000-2019

In [ ]:
def generate_all_recommendations(album_df, album_embeddings, k=10):
    # Full similarity matrix: every album vs. every album
    sims = album_embeddings @ album_embeddings.T
    np.fill_diagonal(sims, -np.inf)  # exclude self-matches
    artist_arr = album_df["artist_norm"].values
    records = []
    for i in range(len(album_df)):
        row_sims = sims[i].copy()
        # Exclude same-artist matches
        same_artist_mask = artist_arr == artist_arr[i]
        row_sims[same_artist_mask] = -np.inf
        top_k_idx = np.argpartition(row_sims, -k)[-k:]
        top_k_idx = top_k_idx[np.argsort(row_sims[top_k_idx])[::-1]]
        for rank, j in enumerate(top_k_idx, 1):
            records.append({
                "query_artist": album_df.iloc[i]["artist_norm"],
                "query_album": album_df.iloc[i]["album_norm"],
                "rec_artist": album_df.iloc[j]["artist_norm"],
                "rec_album": album_df.iloc[j]["album_norm"],
                "rank": rank,
                "score": round(float(row_sims[j]), 4),
                "rec_length_flag": album_df.iloc[j]["length_flag"]
            })
    return pd.DataFrame(records)

all_recs_df = generate_all_recommendations(album_df, album_embeddings, k=10)
print(f"Generated {len(all_recs_df):,} recommendation rows")
all_recs_df.to_parquet(f"{DRIVE_DIR}/recommendations_album_level_avg_embeddings.parquet")

Generated 447,160 recommendation rows


In [ ]:
!pip install rapidfuzz --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.0 MB/s eta 0:00:00


In [ ]:
import re
import unicodedata

def normalize(s):
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("utf-8")
    s = s.lower().strip()
    s = re.sub(r"[^\w\s&]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

print(normalize("Bublé"))  # sanity check: should print buble

buble


In [ ]:
billboard_top10_by_year = [
    (2000, 1, "*NSYNC", "No Strings Attached"),
    (2000, 2, "Santana", "Supernatural"),
    (2000, 3, "Eminem", "The Marshall Mathers LP"),
    (2000, 4, "Britney Spears", "Oops!...I Did It Again"),
    (2000, 6, "Creed", "Human Clay"),
    (2000, 7, "Celine Dion", "All The Way...A Decade Of Song"),
    (2000, 8, "Christina Aguilera", "Christina Aguilera"),
    (2000, 9, "Backstreet Boys", "Millennium"),
    (2000, 10, "DMX", "...And Then There Was X"),
    (2001, 1, "The Beatles", "1"),
    (2001, 2, "Shaggy", "Hotshot"),
    (2001, 3, "Backstreet Boys", "Black & Blue"),
    (2001, 4, "Various Artists", "Now 5"),
    (2001, 5, "Limp Bizkit", "Chocolate Starfish And The Hot Dog Flavored Water"),
    (2001, 6, "Linkin Park", "[Hybrid Theory]"),
    (2001, 7, "Staind", "Break The Cycle"),
    (2001, 8, "Enya", "A Day Without Rain"),
    (2001, 9, "*NSYNC", "Celebrity"),
    (2001, 10, "Nelly", "Country Grammar"),
    (2002, 1, "Eminem", "The Eminem Show"),
    (2002, 2, "Creed", "Weathered"),
    (2002, 3, "Nelly", "Nellyville"),
    (2002, 4, "P!nk", "M!ssundaztood"),
    (2002, 5, "Linkin Park", "[Hybrid Theory]"),
    (2002, 6, "Various Artists", "O Brother, Where Art Thou? Soundtrack"),
    (2002, 7, "Nickelback", "Silver Side Up"),
    (2002, 8, "Britney Spears", "Britney"),
    (2002, 9, "Various Artists", "Now 8"),
    (2002, 10, "Ludacris", "Word Of Mouf"),
    (2003, 1, "50 Cent", "Get Rich Or Die Tryin'"),
    (2003, 2, "Norah Jones", "Come Away With Me"),
    (2003, 3, "Shania Twain", "Up!"),
    (2003, 4, "Dixie Chicks", "Home"),
    (2003, 5, "Avril Lavigne", "Let Go"),
    (2003, 6, "Linkin Park", "Meteora"),
    (2003, 7, "Various Artists", "8 Mile Soundtrack"),
    (2003, 8, "Evanescence", "Fallen"),
    (2003, 9, "Tim McGraw", "Tim McGraw And The Dancehall Doctors"),
    (2003, 10, "Christina Aguilera", "Stripped"),
    (2004, 1, "Usher", "Confessions"),
    (2004, 2, "OutKast", "Speakerboxxx/The Love Below"),
    (2004, 3, "Josh Groban", "Closer"),
    (2004, 4, "Alicia Keys", "The Diary Of Alicia Keys"),
    (2004, 5, "Norah Jones", "Feels Like Home"),
    (2004, 6, "Evanescence", "Fallen"),
    (2004, 7, "Toby Keith", "Shock'n Y'All"),
    (2004, 8, "Britney Spears", "In The Zone"),
    (2004, 9, "Sheryl Crow", "The Very Best Of Sheryl Crow"),
    (2004, 10, "Kenny Chesney", "When The Sun Goes Down"),
    (2005, 1, "50 Cent", "The Massacre"),
    (2005, 2, "Eminem", "Encore"),
    (2005, 3, "Green Day", "American Idiot"),
    (2005, 4, "Mariah Carey", "The Emancipation Of Mimi"),
    (2005, 5, "Kelly Clarkson", "Breakaway"),
    (2005, 6, "Gwen Stefani", "Love. Angel. Music. Baby."),
    (2005, 7, "Destiny's Child", "Destiny Fulfilled"),
    (2005, 8, "U2", "How To Dismantle An Atomic Bomb"),
    (2005, 9, "Shania Twain", "Greatest Hits"),
    (2005, 10, "Rascal Flatts", "Feels Like Today"),
    (2006, 1, "Carrie Underwood", "Some Hearts"),
    (2006, 2, "Various Artists", "High School Musical Soundtrack"),
    (2006, 3, "Nickelback", "All The Right Reasons"),
    (2006, 4, "Rascal Flatts", "Me And My Gang"),
    (2006, 5, "Mary J. Blige", "The Breakthrough"),
    (2006, 6, "Eminem", "Curtain Call: The Hits"),
    (2006, 7, "James Blunt", "Back To Bedlam"),
    (2006, 8, "Kenny Chesney", "The Road And The Radio"),
    (2006, 9, "Johnny Cash", "The Legend Of Johnny Cash"),
    (2006, 10, "Kelly Clarkson", "Breakaway"),
    (2007, 1, "Daughtry", "Daughtry"),
    (2007, 2, "Akon", "Konvicted"),
    (2007, 3, "Fergie", "The Dutchess"),
    (2007, 4, "Various Artists", "Hannah Montana Soundtrack"),
    (2007, 5, "Carrie Underwood", "Some Hearts"),
    (2007, 6, "Nickelback", "All The Right Reasons"),
    (2007, 7, "Justin Timberlake", "FutureSex/LoveSounds"),
    (2007, 8, "Various Artists", "High School Musical 2 Soundtrack"),
    (2007, 9, "Various Artists", "Now 23"),
    (2007, 10, "Linkin Park", "Minutes To Midnight"),
    (2008, 1, "Alicia Keys", "As I Am"),
    (2008, 2, "Josh Groban", "Noel"),
    (2008, 3, "Lil Wayne", "Tha Carter III"),
    (2008, 4, "Eagles", "Long Road Out Of Eden"),
    (2008, 5, "Taylor Swift", "Taylor Swift"),
    (2008, 6, "Kid Rock", "Rock N Roll Jesus"),
    (2008, 7, "Coldplay", "Viva La Vida Or Death And All His Friends"),
    (2008, 8, "Various Artists", "Now 26"),
    (2008, 9, "Carrie Underwood", "Carnival Ride"),
    (2008, 10, "Garth Brooks", "The Ultimate Hits"),
    (2009, 1, "Taylor Swift", "Fearless"),
    (2009, 2, "Beyonce", "I Am...Sasha Fierce"),
    (2009, 3, "Nickelback", "Dark Horse"),
    (2009, 4, "Various Artists", "Twilight Soundtrack"),
    (2009, 5, "Various Artists", "Hannah Montana: The Movie Soundtrack"),
    (2009, 6, "Britney Spears", "Circus"),
    (2009, 7, "Kanye West", "808s & Heartbreak"),
    (2009, 8, "Lady Gaga", "The Fame"),
    (2009, 9, "Eminem", "Relapse"),
    (2009, 10, "The Black Eyed Peas", "The E.N.D."),
    (2010, 1, "Susan Boyle", "I Dreamed A Dream"),
    (2010, 2, "Eminem", "Recovery"),
    (2010, 4, "Lady Gaga", "The Fame"),
    (2010, 5, "Justin Bieber", "My World 2.0"),
    (2010, 6, "Andrea Bocelli", "My Christmas"),
    (2010, 7, "Taylor Swift", "Fearless"),
    (2010, 8, "Justin Bieber", "My World (EP)"),
    (2010, 9, "Taylor Swift", "Speak Now"),
    (2010, 10, "The Black Eyed Peas", "The E.N.D."),
    (2011, 1, "Adele", "21"),
    (2011, 2, "Taylor Swift", "Speak Now"),
    (2011, 3, "Lady Gaga", "Born This Way"),
    (2011, 4, "Jason Aldean", "My Kinda Party"),
    (2011, 5, "Susan Boyle", "The Gift"),
    (2011, 6, "Lil Wayne", "Tha Carter IV"),
    (2011, 7, "Nicki Minaj", "Pink Friday"),
    (2011, 8, "Mumford & Sons", "Sigh No More"),
    (2011, 9, "Rihanna", "LOUD"),
    (2011, 10, "Katy Perry", "Teenage Dream"),
    (2012, 1, "Adele", "21"),
    (2012, 2, "Michael Buble", "Christmas"),
    (2012, 3, "Drake", "Take Care"),
    (2012, 4, "Taylor Swift", "Red"),
    (2012, 5, "One Direction", "Up All Night"),
    (2012, 6, "Luke Bryan", "Tailgates & Tanlines"),
    (2012, 7, "Mumford & Sons", "Babel"),
    (2012, 8, "Rihanna", "Talk That Talk"),
    (2012, 9, "Lionel Richie", "Tuskegee"),
    (2012, 10, "The Black Keys", "El Camino"),
    (2013, 1, "Justin Timberlake", "The 20/20 Experience"),
    (2013, 2, "Taylor Swift", "Red"),
    (2013, 3, "One Direction", "Take Me Home"),
    (2013, 4, "Bruno Mars", "Unorthodox Jukebox"),
    (2013, 5, "Mumford & Sons", "Babel"),
    (2013, 6, "Imagine Dragons", "Night Visions"),
    (2013, 7, "Florida Georgia Line", "Here's To The Good Times"),
    (2013, 8, "P!nk", "The Truth About Love"),
    (2013, 9, "Luke Bryan", "Crash My Party"),
    (2013, 10, "Rihanna", "Unapologetic"),
    (2014, 1, "Various Artists", "Frozen Soundtrack"),
    (2014, 2, "Beyonce", "Beyonce"),
    (2014, 3, "Taylor Swift", "1989"),
    (2014, 4, "One Direction", "Midnight Memories"),
    (2014, 5, "Eminem", "The Marshall Mathers LP 2"),
    (2014, 6, "Lorde", "Pure Heroine"),
    (2014, 7, "Luke Bryan", "Crash My Party"),
    (2014, 8, "Katy Perry", "PRISM"),
    (2014, 9, "Garth Brooks", "Blame It All On My Roots: Five Decades Of Influences"),
    (2014, 10, "Florida Georgia Line", "Here's To The Good Times"),
    (2015, 1, "Taylor Swift", "1989"),
    (2015, 2, "Ed Sheeran", "x"),
    (2015, 3, "Sam Smith", "In The Lonely Hour"),
    (2015, 4, "Drake", "If You're Reading This It's Too Late"),
    (2015, 5, "Meghan Trainor", "Title"),
    (2015, 6, "Maroon 5", "V"),
    (2015, 7, "Nicki Minaj", "The Pinkprint"),
    (2015, 8, "J. Cole", "2014 Forest Hills Drive"),
    (2015, 9, "Various Artists", "Fifty Shades Of Grey Soundtrack"),
    (2015, 10, "One Direction", "FOUR"),
    (2016, 1, "Adele", "25"),
    (2016, 2, "Drake", "Views"),
    (2016, 3, "Justin Bieber", "Purpose"),
    (2016, 4, "Beyonce", "Lemonade"),
    (2016, 5, "Rihanna", "ANTI"),
    (2016, 6, "Twenty One Pilots", "Blurryface"),
    (2016, 7, "Chris Stapleton", "Traveller"),
    (2016, 8, "One Direction", "Made In The A.M."),
    (2016, 9, "The Weeknd", "Beauty Behind The Madness"),
    (2016, 10, "Original Broadway Cast", "Hamilton: An American Musical"),
    (2017, 1, "Kendrick Lamar", "DAMN."),
    (2017, 2, "Bruno Mars", "24K Magic"),
    (2017, 3, "The Weeknd", "Starboy"),
    (2017, 4, "Ed Sheeran", "÷ (Divide)"),
    (2017, 5, "Drake", "More Life"),
    (2017, 6, "Various Artists", "Moana Soundtrack"),
    (2017, 7, "Post Malone", "Stoney"),
    (2017, 8, "Migos", "Culture"),
    (2017, 9, "Original Broadway Cast", "Hamilton: An American Musical"),
    (2017, 10, "J. Cole", "4 Your Eyez Only"),
    (2018, 1, "Taylor Swift", "reputation"),
    (2018, 2, "Drake", "Scorpion"),
    (2018, 3, "Post Malone", "beerbongs & bentleys"),
    (2018, 4, "Various Artists", "The Greatest Showman Soundtrack"),
    (2018, 5, "Ed Sheeran", "÷ (Divide)"),
    (2018, 6, "Cardi B", "Invasion Of Privacy"),
    (2018, 7, "Travis Scott", "ASTROWORLD"),
    (2018, 8, "Post Malone", "Stoney"),
    (2018, 9, "XXXTENTACION", "?"),
    (2018, 10, "Migos", "Culture II"),
    (2019, 1, "Billie Eilish", "When We All Fall Asleep, Where Do We Go?"),
    (2019, 2, "Ariana Grande", "Thank U, Next"),
    (2019, 3, "Lady Gaga & Bradley Cooper", "A Star Is Born Soundtrack"),
    (2019, 4, "Taylor Swift", "Lover"),
    (2019, 5, "Post Malone", "beerbongs & bentleys"),
    (2019, 6, "Drake", "Scorpion"),
    (2019, 7, "Meek Mill", "Championships"),
    (2019, 8, "Travis Scott", "ASTROWORLD"),
    (2019, 9, "Post Malone", "Hollywood's Bleeding"),
    (2019, 10, "A Boogie wit da Hoodie", "Hoodie SZN"),
]

bb_df = pd.DataFrame(billboard_top10_by_year, columns=["year", "rank", "artist", "album"])
bb_df["bb_artist_norm"] = bb_df["artist"].apply(normalize)
bb_df["bb_album_norm"] = bb_df["album"].apply(normalize)

album_df["artist_norm_clean"] = album_df["artist_norm"].apply(normalize)
album_df["album_norm_clean"] = album_df["album_norm"].apply(normalize)

exact = bb_df.merge(
    album_df,
    left_on=["bb_artist_norm", "bb_album_norm"],
    right_on=["artist_norm_clean", "album_norm_clean"],
    how="left",
    indicator=True
)
matched_exact = exact[exact["_merge"] == "both"]
unmatched = exact[exact["_merge"] == "left_only"][["year", "rank", "artist", "album", "bb_artist_norm", "bb_album_norm"]]

print(f"Exact matches: {len(matched_exact)} / {len(bb_df)}")
print(f"Unmatched, going to fuzzy stage: {len(unmatched)}")

fuzzy_results = []
for _, row in unmatched.iterrows():
    artist_choices = album_df["artist_norm_clean"].unique()
    best_artist, artist_score, _ = process.extractOne(
        row["bb_artist_norm"], artist_choices, scorer=fuzz.WRatio
    )

    if artist_score < 80:
        fuzzy_results.append({**row, "matched_artist": None, "matched_album": None,
                               "artist_score": artist_score, "album_score": None})
        continue

    candidate_albums = album_df[album_df["artist_norm_clean"] == best_artist]["album_norm_clean"]
    if candidate_albums.empty:
        continue

    best_album, album_score, _ = process.extractOne(
        row["bb_album_norm"], candidate_albums, scorer=fuzz.WRatio
    )

    fuzzy_results.append({
        **row,
        "matched_artist": best_artist,
        "matched_album": best_album,
        "artist_score": artist_score,
        "album_score": album_score
    })

fuzzy_df = pd.DataFrame(fuzzy_results)
print(f"\nFuzzy candidates found: {fuzzy_df['matched_album'].notna().sum()}")
print(fuzzy_df[["year", "artist", "album", "matched_artist", "matched_album", "artist_score", "album_score"]]
      .sort_values("album_score"))

Exact matches: 116 / 198
Unmatched, going to fuzzy stage: 84

Fuzzy candidates found: 84
    year           artist                                  album  \
63  2013       Luke Bryan                         Crash My Party   
82  2018     XXXTENTACION                                      ?   
70  2015       Ed Sheeran                                      x   
66  2014       Luke Bryan                         Crash My Party   
6   2000  Backstreet Boys                             Millennium   
..   ...              ...                                    ...   
49  2008  Various Artists                                 Now 26   
39  2006      Johnny Cash              The Legend Of Johnny Cash   
56  2010    Justin Bieber                          My World (EP)   
19  2002  Various Artists  O Brother, Where Art Thou? Soundtrack   
51  2009          Beyonce                    I Am...Sasha Fierce   

     matched_artist                                 matched_album  \
63               lu      

In [ ]:
# --- Combine exact matches + high-confidence fuzzy matches (>=90) into final set ---

# Exact matches: keep the billboard fields + matched album_df fields
exact_final = matched_exact.drop(columns=["_merge"]).copy()
exact_final["match_type"] = "exact"

# High-confidence fuzzy matches: re-join back to album_df to get full album record
fuzzy_accept = fuzzy_df[fuzzy_df["album_score"] >= 90].copy()
fuzzy_accept = fuzzy_accept.merge(
    album_df,
    left_on=["matched_artist", "matched_album"],
    right_on=["artist_norm_clean", "album_norm_clean"],
    how="left"
)
fuzzy_accept["match_type"] = "fuzzy"

# Align columns between the two sets before concatenating
common_cols = [c for c in exact_final.columns if c in fuzzy_accept.columns]
final_matched = pd.concat(
    [exact_final[common_cols], fuzzy_accept[common_cols]],
    ignore_index=True
)

print(f"Final matched set: {len(final_matched)} / {len(bb_df)} Billboard top-10 entries")
print(f"  Exact: {len(exact_final)}")
print(f"  Fuzzy (score >= 90): {len(fuzzy_accept)}")
print(f"  Dropped (no confident match): {len(bb_df) - len(final_matched)}")

final_matched.head(20)

Final matched set: 127 / 198 Billboard top-10 entries
  Exact: 116
  Fuzzy (score >= 90): 11
  Dropped (no confident match): 71


,year,rank,artist,album,bb_artist_norm,bb_album_norm,artist_norm,album_norm,n_reviews,sources,length_flag,artist_norm_clean,album_norm_clean,match_type
0,2000,3,Eminem,The Marshall Mathers LP,eminem,the marshall mathers lp,eminem,the marshall mathers lp,1.0,[pitchfork],False,eminem,the marshall mathers lp,exact
1,2001,9,*NSYNC,Celebrity,nsync,celebrity,*nsync,celebrity,1.0,[critique_brainz],False,nsync,celebrity,exact
2,2001,10,Nelly,Country Grammar,nelly,country grammar,nelly,country grammar,1.0,[pitchfork],False,nelly,country grammar,exact
3,2002,1,Eminem,The Eminem Show,eminem,the eminem show,eminem,the eminem show,2.0,"[critique_brainz, pitchfork]",False,eminem,the eminem show,exact
4,2002,4,P!nk,M!ssundaztood,pnk,mssundaztood,p!nk,m!ssundaztood,1.0,[critique_brainz],False,pnk,mssundaztood,exact
5,2003,1,50 Cent,Get Rich Or Die Tryin',50 cent,get rich or die tryin,50 cent,get rich or die tryin',1.0,[pitchfork],False,50 cent,get rich or die tryin,exact
6,2003,1,50 Cent,Get Rich Or Die Tryin',50 cent,get rich or die tryin,50 cent,get rich or die tryin’,1.0,[critique_brainz],False,50 cent,get rich or die tryin,exact
7,2003,2,Norah Jones,Come Away With Me,norah jones,come away with me,norah jones,come away with me,1.0,[critique_brainz],False,norah jones,come away with me,exact
8,2003,4,Dixie Chicks,Home,dixie chicks,home,dixie chicks,home,2.0,"[critique_brainz, pitchfork]",True,dixie chicks,home,exact
9,2003,5,Avril Lavigne,Let Go,avril lavigne,let go,avril lavigne,let go,2.0,"[critique_brainz, pitchfork]",True,avril lavigne,let go,exact


In [ ]:
# --- Filter full recommendation database to Billboard top-10 query albums ---

all_recs_df = pd.read_parquet(f"{DRIVE_DIR}/recommendations_album_level_avg_embeddings.parquet")

billboard_keys = final_matched[["artist_norm_clean", "album_norm_clean"]].drop_duplicates()
billboard_keys = billboard_keys.rename(columns={
    "artist_norm_clean": "query_artist",
    "album_norm_clean": "query_album"
})

billboard_full_recs = all_recs_df.merge(
    billboard_keys, on=["query_artist", "query_album"], how="inner"
)

print(f"Billboard query albums found in full rec database: {billboard_full_recs['query_album'].nunique()} / {len(billboard_keys)}")
print(f"Total recommendation rows for handoff: {len(billboard_full_recs)}")

billboard_full_recs.to_parquet(f"{DRIVE_DIR}/billboard_top10_recommendations_for_comparison.parquet")
billboard_full_recs.to_csv(f"{DRIVE_DIR}/billboard_top10_recommendations_for_comparison.csv", index=False)

billboard_full_recs.head(20)

Billboard query albums found in full rec database: 80 / 108
Total recommendation rows for handoff: 800


,query_artist,query_album,rec_artist,rec_album,rank,score,rec_length_flag
0,50 cent,the massacre,various artists,get rich or die tryin' ost,1,0.8955,False
1,50 cent,the massacre,the game,the documentary,2,0.7946,False
2,50 cent,the massacre,mobb deep,blood money,3,0.7943,False
3,50 cent,the massacre,mf grimm,the downfall of ibliys: a ghetto opera,4,0.7873,False
4,50 cent,the massacre,killer mike,ghetto extraordinary,5,0.7865,False
5,50 cent,the massacre,the game,doctor's advocate,6,0.7832,False
6,50 cent,the massacre,n.w.a,straight outta compton,7,0.7761,False
7,50 cent,the massacre,chief keef,finally rich,8,0.7708,False
8,50 cent,the massacre,gunplay,601 & snort,9,0.7704,False
9,50 cent,the massacre,public enemy,new whirl odor,10,0.7669,True


### Recommendation "chain"

Generates a chain of recommendations based on the top previously recommended album.

In [2]:
def generate_recommendation_chain(seed_artist_norm, seed_album_norm, album_df, album_embeddings, chain_length=10):
    """Generate a chain: seed -> rec1 -> rec2 -> ... by always stepping to the
    closest unvisited album. Excludes both prior albums AND prior artists."""
    seed_mask = (album_df["artist_norm"] == seed_artist_norm) & (album_df["album_norm"] == seed_album_norm)
    if not seed_mask.any():
        print(f"Not found: {seed_artist_norm} — {seed_album_norm}")
        return None

    current_idx = album_df[seed_mask].index[0]
    visited_artists = {seed_artist_norm}
    chain = [{
        "step": 0,
        "artist": seed_artist_norm,
        "album": seed_album_norm,
        "score": None  # seed has no similarity score to itself
    }]

    for step in range(1, chain_length + 1):
        current_vec = album_embeddings[current_idx]
        sims = album_embeddings @ current_vec
        sims[current_idx] = -np.inf

        artist_arr = album_df["artist_norm"].values
        visited_mask = np.isin(artist_arr, list(visited_artists))
        sims[visited_mask] = -np.inf

        next_idx = np.argmax(sims)
        if sims[next_idx] == -np.inf:
            print(f"Chain ended early at step {step}: no unvisited artists remain")
            break

        next_artist = album_df.loc[next_idx, "artist_norm"]
        next_album = album_df.loc[next_idx, "album_norm"]

        chain.append({
            "step": step,
            "artist": next_artist,
            "album": next_album,
            "score": round(float(sims[next_idx]), 4)
        })

        visited_artists.add(next_artist)
        current_idx = next_idx

    return pd.DataFrame(chain)

In [7]:
# Testing function

chain = generate_recommendation_chain("radiohead", "kid a", album_df, album_embeddings, chain_length=10)
chain

,step,artist,album,score
0,0,radiohead,kid a,NaN
1,1,endochine,day two,0.8478
2,2,thom yorke,the eraser,0.8185
3,3,atoms for peace,amok,0.8784
4,4,eob,earth,0.8187
5,5,earth,"angels of darkness, demons of light 1",0.8130
6,6,the bug vs earth,concrete desert,0.8232
7,7,the bug,angels & devils,0.8134
8,8,king midas sound,waiting for you…,0.8676
9,9,kingdom,that mystic ep,0.8585


In [8]:
def generate_all_chains(album_df, album_embeddings, chain_length=10):
    """Run generate_recommendation_chain for every album in the dataset.
    Returns a long-format dataframe: one row per (seed, step)."""
    artist_arr = album_df["artist_norm"].values
    album_arr = album_df["album_norm"].values
    n = len(album_df)

    all_records = []

    for seed_idx in range(n):
        seed_artist = artist_arr[seed_idx]
        seed_album = album_arr[seed_idx]

        current_idx = seed_idx
        visited_artists = {seed_artist}

        all_records.append({
            "seed_artist": seed_artist,
            "seed_album": seed_album,
            "step": 0,
            "artist": seed_artist,
            "album": seed_album,
            "score": None
        })

        for step in range(1, chain_length + 1):
            current_vec = album_embeddings[current_idx]
            sims = album_embeddings @ current_vec
            sims[current_idx] = -np.inf

            visited_mask = np.isin(artist_arr, list(visited_artists))
            sims[visited_mask] = -np.inf

            next_idx = np.argmax(sims)
            if sims[next_idx] == -np.inf:
                break

            next_artist = artist_arr[next_idx]
            next_album = album_arr[next_idx]

            all_records.append({
                "seed_artist": seed_artist,
                "seed_album": seed_album,
                "step": step,
                "artist": next_artist,
                "album": next_album,
                "score": round(float(sims[next_idx]), 4)
            })

            visited_artists.add(next_artist)
            current_idx = next_idx

        if seed_idx % 2000 == 0:
            print(f"  {seed_idx:,} / {n:,} seeds processed")

    return pd.DataFrame(all_records)

all_chains_df = generate_all_chains(album_df, album_embeddings, chain_length=10)
print(f"\nGenerated {len(all_chains_df):,} chain-step rows across {all_chains_df['seed_album'].nunique():,} seed albums")

all_chains_df.to_parquet(f"{DRIVE_DIR}/recommendation_chains_all_albums.parquet")

  0 / 44,716 seeds processed
  2,000 / 44,716 seeds processed
  4,000 / 44,716 seeds processed
  6,000 / 44,716 seeds processed
  8,000 / 44,716 seeds processed
  10,000 / 44,716 seeds processed
  12,000 / 44,716 seeds processed
  14,000 / 44,716 seeds processed
  16,000 / 44,716 seeds processed
  18,000 / 44,716 seeds processed
  20,000 / 44,716 seeds processed
  22,000 / 44,716 seeds processed
  24,000 / 44,716 seeds processed
  26,000 / 44,716 seeds processed
  28,000 / 44,716 seeds processed
  30,000 / 44,716 seeds processed
  32,000 / 44,716 seeds processed
  34,000 / 44,716 seeds processed
  36,000 / 44,716 seeds processed
  38,000 / 44,716 seeds processed
  40,000 / 44,716 seeds processed
  42,000 / 44,716 seeds processed
  44,000 / 44,716 seeds processed

Generated 491,876 chain-step rows across 42,039 seed albums
